# DreamPrice Quick Start

This notebook walks through a minimal end-to-end example: create a pricing
environment, run a rollout, and visualize the results.

## Install
```bash
pip install torch --index-url https://download.pytorch.org/whl/cu121
pip install mamba-ssm causal-conv1d
pip install -e '.[dev,docs]'
```

In [ ]:
from retail_world_model import WorldModel, RLAgent, PricingEnvironment, Trainer
import numpy as np
import matplotlib.pyplot as plt

print(f'Imports OK')

In [ ]:
# Create a pricing environment with 3 SKUs, 13-week episodes
env = PricingEnvironment(n_skus=3, episode_length=13)
obs, info = env.reset(seed=42)
print(f'Observation shape: {obs.shape}')
print(f'Action space: {env.action_space}')

In [ ]:
# Run a 13-step rollout with random actions
prices = [info.get('prices', np.ones(3))]
rewards = []
obs, info = env.reset(seed=42)

for step in range(13):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    prices.append(info.get('prices', np.ones(3)))
    rewards.append(reward)
    if terminated or truncated:
        break

prices = np.array(prices)
rewards = np.array(rewards)
print(f'Episode length: {len(rewards)} steps')
print(f'Total reward: {rewards.sum():.2f}')

In [ ]:
# Plot price trajectories for each SKU
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for sku in range(prices.shape[1]):
    axes[0].plot(prices[:, sku], label=f'SKU {sku}')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price Trajectories')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(rewards)), rewards, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Week')
axes[1].set_ylabel('Reward')
axes[1].set_title('Weekly Rewards')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Episode reward breakdown
print('Episode Summary')
print('=' * 40)
print(f'Total reward:   {rewards.sum():>10.2f}')
print(f'Mean reward:    {rewards.mean():>10.2f}')
print(f'Std reward:     {rewards.std():>10.2f}')
print(f'Min reward:     {rewards.min():>10.2f}')
print(f'Max reward:     {rewards.max():>10.2f}')
print(f'Final prices:   {prices[-1]}')